## **Preparing reference datasets for OLMO Earth: DS3 (crop type) & DS4 (cropland status)**

This notebook is workshop material for the [OLMO Earth](https://olmoearth.allenai.org/) data-prep session. It reproduces, from two public GitHub repositories, two of the point-based reference datasets used to train Ukraine winter-crop models: how the raw multi-year field/photo-interpreted annotations were reclassified into the class schemes the models were trained on, and re-formatted to match OLMO Earth's expected input format.

Author: Josef Wagner ([University of Strasbourg](https://www.unistra.fr/fr), [NASA Harvest](https://www.nasaharvest.org/)) jwagner@unistra.fr

### **Context**

OLMO Earth trains on point-referenced labels: a longitude/latitude/time triplet plus a class. Our raw annotation data isn't in that shape or class scheme -- it comes as a wide-format shapefile with one column per annotation year, or as yearly GeoPackages with their own class vocabulary, each built for a different purpose (monitoring cropland status vs. monitoring crop type). Getting from "raw annotations" to "OLMO-ready training set" means picking, for each target task, which raw classes matter and how they collapse together -- that reclassification step is the core of this notebook.

### **What this notebook produces**

- **DS4** -- cropland status (Cropland / Non-cropland / Abandoned / Fallow), reproduced from the `Ukraine_CIS_annotations_2021-2024` repo (2021-2024 seasons).
- **DS3** -- crop type on confirmed cropland (Winter cereals / Rapeseed / Summer crop), reproduced from the `Uk_sample_units_22-25` repo (2022-2025 seasons).

> The production versions of DS4 (and its crop-type counterpart DS5) used operationally also blend in 2026 stac-annotator annotations that live outside these two repos. This notebook intentionally reproduces the repo-sourced core only, to keep a clean "two repos in -> two datasets out" story.

### **Source repos**

- CIS annotations 2021-2024: https://github.com/jowa-ea/Ukraine_CIS_annotations_2021-2024 -- Wagner, J. et al., *Monitoring cropland cultivation, abandonment, fallowing and recultivation dynamics with regard to conflict intensity in war-affected Ukraine*, Science of Remote Sensing, 12 (2025). https://doi.org/10.1016/j.srs.2025.100326
- Annotated sample units 2022-2025: https://github.com/jowa-ea/Uk_sample_units_22-25 -- Wagner, J., Skakun, S., Nair, S.S. et al., *Monitoring winter crop areas during wartime: remote sensing support for Ukraine's agricultural statistics*, npj Sustainable Agriculture, 4, 1 (2026). https://doi.org/10.1038/s44264-025-00119-4

### **Setup**

This notebook runs the same way locally and in Colab. Locally, create a Python environment, run `pip install -r requirements.txt`, and run from this folder. In Colab, the next cells detect the environment, install the missing packages, and clone this repo (`OlmoEarthWalkthrough`) so the notebook has access to its own `pipeline_utils.py` and the bundled oblast boundary file under `data/`.

In [ ]:
def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if in_colab():
    # pandas & matplotlib ship with Colab; geopandas/shapely do not.
    %pip install -q geopandas shapely
else:
    print("Running locally - make sure you've installed the dependencies "
          "(pip install -r requirements.txt).")

In [ ]:
import subprocess
from pathlib import Path

WORKSHOP_REPO_URL = "https://github.com/jowa-ea/OlmoEarthWalkthrough.git"
WORKSHOP_REPO_NAME = "OlmoEarthWalkthrough"

if in_colab():
    if not Path(WORKSHOP_REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", WORKSHOP_REPO_URL], check=True)
    import os
    os.chdir(WORKSHOP_REPO_NAME)
    print(f"Working directory: {Path.cwd()}")
else:
    print("Running locally - skipping self-clone, using this notebook's own directory.")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

try:
    import pipeline_utils as pu
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "pipeline_utils.py not found - make sure you're running this notebook "
        "from the AI2_workshop_olmo_walkthrough folder."
    ) from exc

pd.options.display.float_format = "{:.2f}".format

### **Available data**

Both source repos hold point annotations for Ukraine cropland monitoring, but with different schemas:

- **`Ukraine_CIS_annotations_2021-2024`** -- a single shapefile, one row per point, with one class column per year (`Label_21` ... `Label_24`, wide format). Four classes: `Cultivated`, `Non-crop`, `Abandoned`, `Fallow`.
- **`Uk_sample_units_22-25`** -- one GeoPackage per year (2022-2025, long format: one row per point per year), with a `Class_st` column whose vocabulary shifted slightly over time (crop types plus, from 2024, a `Not_winter_cro` catch-all).

In [ ]:
# In Colab we already chdir'd into the cloned repo above; locally we assume
# the notebook is being run from its own folder. Either way cwd is the root.
BASE_DIR = Path.cwd()

REPOS_DIR = BASE_DIR / "workshop_prep" / "repos"
OUTPUT_DIR = BASE_DIR / "olmo_trainsets"
BOUNDARY_PATH = BASE_DIR / "data" / "oblasts_simplified.geojson"

repos = pu.clone_data_repos(REPOS_DIR)
CIS_SHP = repos["Ukraine_CIS_annotations_2021-2024"] / "CIS_annotations_2021-2024.shp"
GPKG_DIR = repos["Uk_sample_units_22-25"]

## Step 1 -- Load raw data

We load both sources into a common long shape (`longitude, latitude, year, original_class`, plus `fid` for the GeoPackages) without touching the class values yet -- that keeps the raw exploration below honest to what the annotators actually recorded.

In [ ]:
cis_raw = pu.read_cis(CIS_SHP)
print(f"{len(cis_raw)} point-year records loaded from the CIS shapefile")
cis_raw.head()

In [ ]:
gpkg_raw = pu.load_gpkgs(GPKG_DIR)
print(f"{len(gpkg_raw)} points loaded from the Uk_sample_units GPKGs (2022-2025)")
gpkg_raw.head()

## Step 2 -- Explore the raw data: spatial distribution

Before reclassifying anything, it's worth checking *where* the annotations are and whether their spatial spread looks reasonable across oblasts (uneven coverage would bias any model trained on them). We plot one map per year, since coverage and land-use conditions shift as the war progresses.

In [ ]:
oblasts = gpd.read_file(BOUNDARY_PATH)


def plot_spatial_distribution(df: pd.DataFrame, title: str, class_col: str = "original_class") -> None:
    """One map per year (small multiples), points colored by class_col, shared legend."""
    gdf = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326"
    )
    years = sorted(gdf["year"].unique())
    classes = sorted(gdf[class_col].unique())
    cmap = plt.get_cmap("tab10")
    colors = {cls: cmap(i % 10) for i, cls in enumerate(classes)}

    ncols = 2
    nrows = -(-len(years) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows + 0.6), squeeze=False)
    axes = axes.flatten()

    for ax, year in zip(axes, years):
        oblasts.boundary.plot(ax=ax, color="lightgrey", linewidth=0.5)
        year_gdf = gdf[gdf["year"] == year]
        for cls, sub in year_gdf.groupby(class_col):
            sub.plot(ax=ax, markersize=6, alpha=0.6, color=colors[cls])
        ax.set_title(str(year))
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for ax in axes[len(years):]:
        ax.set_visible(False)

    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=colors[cls], label=str(cls))
        for cls in classes
    ]
    fig.suptitle(title)
    fig.tight_layout(rect=[0, 0.08, 1, 0.95])
    fig.legend(handles=handles, title=class_col, loc="lower center", ncol=len(classes), bbox_to_anchor=(0.5, 0.0))
    plt.show()


plot_spatial_distribution(cis_raw, "CIS annotations 2021-2024 - spatial distribution (raw classes)")

In [ ]:
plot_spatial_distribution(gpkg_raw, "Uk_sample_units 2022-2025 - spatial distribution (raw classes)")

## Step 2 (continued) -- Yearly label distribution

Class balance shifts year to year -- war-driven land-use change (abandonment, occupation) means the raw label mix isn't stationary. Worth seeing before we start dropping or merging classes.

In [ ]:
def plot_yearly_distribution(df: pd.DataFrame, title: str, class_col: str = "original_class") -> None:
    counts = df.groupby(["year", class_col]).size().unstack(fill_value=0)
    ax = counts.plot(kind="bar", stacked=True, figsize=(8, 5))
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of samples")
    ax.legend(title=class_col, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()


plot_yearly_distribution(cis_raw, "CIS annotations 2021-2024 - yearly label distribution (raw classes)")

In [ ]:
plot_yearly_distribution(gpkg_raw, "Uk_sample_units 2022-2025 - yearly label distribution (raw classes)")

## Step 3 -- Reclassification logic

Each raw source is annotated for its own purpose and has its own class vocabulary. To get an OLMO-ready training set for a *specific* task, we map (and sometimes drop) raw classes into a small, task-relevant scheme. The tables below are the actual mapping applied in code right after (`pipeline_utils.DS4_ENCODING` / `DS3_ENCODING`) -- with the reasoning for each choice.

### DS4 -- cropland status (from `Ukraine_CIS_annotations_2021-2024`)

| Original class | OLMO class | OLMO label | Why |
|---|---|---|---|
| `Cultivated` | 1 | Cropland | Actively cultivated agricultural land -- the core "cropland" signal the model needs to learn. |
| `Non-crop` | 2 | Non-cropland | Land never in agricultural use -- the negative class for the crop/non-crop task. |
| `Abandoned` | 3 | Abandoned | Formerly cultivated land taken out of production (often war-related). Kept as its own class rather than merged into non-cropland, so the model can distinguish long-term non-crop land from recently abandoned cropland -- a policy-relevant distinction for wartime monitoring. |
| `Fallow` | 4 | Fallow | Cropland resting for a season under normal crop rotation. Kept distinct from "abandoned" because fallow land is expected to return to production and has a different spectral/temporal signature. |

### DS3 -- crop type (from `Uk_sample_units_22-25`, cropland only)

| Original class | OLMO class | OLMO label | Why |
|---|---|---|---|
| `Winter_cer` | 1 | Winter cereals | Primary crop type of interest for winter-crop area estimation (wheat, barley, etc. sown in autumn). |
| `Rapeseed` | 2 | Rapeseed | Winter oilseed, spectrally/phenologically distinct from cereals but grown in the same winter-crop season -- needs its own class so the model doesn't conflate the two. |
| `Summer_cro` | 3 | Summer crop | Included so the classifier learns a genuine "not winter crop but still cropland" contrast, rather than only ever seeing winter-crop vs. non-crop. Only annotated in 2022-2023. |
| `Not_winter_cro` (2024-2025 catch-all) | *dropped* | -- | Introduced in the 2024/25 annotation rounds for non-winter-crop cropland that wasn't specifically re-labeled; doesn't map cleanly to any of the three target classes and would add label noise. |
| `Non-crop` / other background classes | *dropped* | -- | DS3 is restricted to confirmed cropland with a known type -- background/non-crop classes belong to DS1/DS2/DS4, not the crop-type dataset. |

In [ ]:
print("DS4 encoding:", pu.DS4_ENCODING)
print("DS4 labels:  ", pu.DS4_LABELS)
print()
print("DS3 encoding:", pu.DS3_ENCODING)
print("DS3 labels:  ", pu.DS3_LABELS)

## Step 4 -- Build DS4

`pu.build_ds4` applies the `DS4_ENCODING` mapping above, adds the `time` column (`{year}-04-15`, per-year annotation date), and raises if it encounters a class outside the mapping (a sign the source data changed).

In [ ]:
ds4 = pu.build_ds4(cis_raw)
print("DS4 class distribution:")
print(ds4["label_cropland_full"].value_counts())
ds4.head()

In [ ]:
plot_spatial_distribution(ds4, "DS4 - cropland status - spatial distribution (reclassified)", class_col="label_cropland_full")
plot_yearly_distribution(ds4, "DS4 - cropland status - yearly distribution (reclassified)", class_col="label_cropland_full")

## Step 5 -- Build DS3

`pu.build_ds3` filters `gpkg_raw` down to the three classes in `DS3_ENCODING` (everything else, including `Not_winter_cro`, is dropped) and applies the mapping.

In [ ]:
dropped = gpkg_raw.loc[~gpkg_raw["original_class"].isin(pu.DS3_ENCODING), "original_class"].value_counts()
print("Classes dropped from DS3 (not cropland-with-known-type):")
print(dropped)

ds3 = pu.build_ds3(gpkg_raw)
print("\nDS3 class distribution:")
print(ds3["label_crop_type"].value_counts())
ds3.head()

In [ ]:
plot_spatial_distribution(ds3, "DS3 - crop type - spatial distribution (reclassified)", class_col="label_crop_type")
plot_yearly_distribution(ds3, "DS3 - crop type - yearly distribution (reclassified)", class_col="label_crop_type")

## Step 6 -- OLMO input format & export

OLMO Earth accepts two file formats for reference/training data:

**CSV (`.csv`)**
- Must contain separate columns for `longitude`, `latitude`, and `time`.
- `time` must be a standardized date/time format, e.g. `2026-03-10T20:09:21.193Z` or `2026-03-10`. If no timezone is given, UTC is assumed.
- Max 100,000 rows per file; larger datasets must be split into multiple files.

**GeoJSON (`.geojson`, `.json`)**
- Must be a `FeatureCollection` of `Feature`s with `Point`, `Polygon`, or `MultiPolygon` geometry.
- Each feature's properties must include `time` in the same standardized format as above.
- Max 100,000 features per file; split into multiple files if larger.

Both DS3 (~2,256 records) and DS4 (8,000 records) fit comfortably under the 100k-row/feature limit, so each is written as a single `_part001` file here -- `pu.export_csv` / `pu.export_geojson` still chunk automatically in case that changes. `pu.validate_olmo_format` checks the required columns and the `time` format before we write anything.

In [ ]:
DS4_COLUMNS = ["longitude", "latitude", "time", "year", "original_class",
               "class_cropland_full", "label_cropland_full"]
DS3_COLUMNS = ["longitude", "latitude", "time", "year", "fid", "original_class",
               "class_crop_type", "label_crop_type"]

pu.validate_olmo_format(ds4)
pu.validate_olmo_format(ds3)

ds4_paths = pu.export_csv(ds4, OUTPUT_DIR, "ds4_cropland_full_21-24", DS4_COLUMNS)
ds4_paths += pu.export_geojson(ds4, OUTPUT_DIR, "ds4_cropland_full_21-24", DS4_COLUMNS)
ds3_paths = pu.export_csv(ds3, OUTPUT_DIR, "ds3_crop_types_22-25", DS3_COLUMNS)
ds3_paths += pu.export_geojson(ds3, OUTPUT_DIR, "ds3_crop_types_22-25", DS3_COLUMNS)

print("Written to", OUTPUT_DIR, ":")
for p in ds4_paths + ds3_paths:
    print(" ", p.name)

## Wrap-up

Starting from two public annotation repos, we:
1. loaded the raw, differently-shaped annotation sources into a common point/year table,
2. looked at their spatial and yearly label distributions before touching anything,
3. reclassified each source into a small, task-specific class scheme with an explicit rationale per class, and
4. exported both datasets in OLMO's CSV/GeoJSON format, chunked and validated against its column/row-limit requirements.

`olmo_trainsets/` now holds the repo-only DS3 and DS4. For the production pipeline -- which additionally folds in 2026 stac-annotator annotations (DS4 -> full DS4, DS3 -> DS5) -- see the equivalent scripts in the broader Ukraine winter-crop mapping pipeline (not part of this public repo). The same pipeline used here also runs headless via `main.py`, for a quick local re-run without the notebook.